# Experimento 3 - Drop Insulin e SkinThickness, imputação por mediana e RobustScaler

## Dataset de treino

In [5]:
import pandas as pd
from sklearn.preprocessing import RobustScaler

NORMALIZE_COLS = ['Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction']

df = pd.read_csv("../../diabetes_dataset.csv")
df.shape


(572, 9)

### 1. Dropando `Insulin` e `SkinThickness`

65% dos registros não tinham o valor de Insulin e cerca de 40% não tinham o valor de SkinThickness, e, ao tentar imputar por mediana, criava um pico artificial enorme na coluna, então ela sai do dataset.

In [6]:
df = df.drop(columns=['Insulin'])
df = df.drop(columns=['SkinThickness'])
df.columns.tolist()


['Pregnancies',
 'Glucose',
 'BloodPressure',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age',
 'Outcome']

### 2. Imputando NaN com a mediana

Guardamos as medianas calculadas aqui — elas também vão ser usadas no
dataset de previsão, pra não usar nenhuma estatística fora do treino.

In [7]:
medianas = df.median(numeric_only=True)
df = df.fillna(medianas)
df.isna().sum()

# medianas

Pregnancies                 0
Glucose                     0
BloodPressure               0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

### 3. Normalizando com RobustScaler

Só em `Glucose`, `BloodPressure`, `BMI`, `DiabetesPedigreeFunction`. `Pregnancies` e `Age` ficam brutas (são contagens/anos, não  contínuas como as outras), e `Outcome` não é tocado. O `scaler` é ajustado aqui e reaproveitado no dataset de previsão.

In [8]:
scaler = RobustScaler()
df[NORMALIZE_COLS] = scaler.fit_transform(df[NORMALIZE_COLS])
df.head()


,Pregnancies,Glucose,BloodPressure,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,0.789809,0.000000,0.181818,0.764547,50,1
1,1,-0.815287,-0.428571,-0.613636,0.017591,31,0
2,8,1.681529,-0.571429,-0.988636,0.886333,32,1
3,0,0.509554,-2.285714,1.261364,5.259811,33,1
4,5,-0.025478,0.142857,-0.727273,-0.388363,30,0


### 4. Exportando

In [9]:
df.to_csv("diabetes-tratado-ex6.csv", index=False)
print("treino exportado com", len(df), "registros e colunas:", list(df.columns))


treino exportado com 572 registros e colunas: ['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']


## Dataset de previsão (`diabetes_app.csv`)

Drop de `Insulin` e `SkinThickness` para consistência com o dataset de treino, e usamos o mesmo `scaler` já ajustados acima (sem `fit` de novo) — treino e previsão precisam ficar na mesma escala pro KNN funcionar.

In [10]:
df_app = pd.read_csv("../../diabetes_app.csv")
df_app = df_app.drop(columns=['Insulin'])
df_app = df_app.drop(columns=['SkinThickness'])
df_app[NORMALIZE_COLS] = scaler.transform(df_app[NORMALIZE_COLS])
df_app.head()


,Pregnancies,Glucose,BloodPressure,BMI,DiabetesPedigreeFunction,Age
0,3,0.305732,-0.571429,-0.636364,-0.339648,28
1,3,1.885350,-0.285714,-0.125000,-0.123139,34
2,15,0.484076,-0.142857,0.579545,-0.518268,43
3,1,-0.203822,-0.857143,-0.750000,1.630582,21
4,8,0.866242,0.428571,1.238636,0.464141,36


In [11]:
df_app.to_csv("diabetes_app-tratado-ex6.csv", index=False)
print("app exportado com", len(df_app), "registros e colunas:", list(df_app.columns))


app exportado com 196 registros e colunas: ['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction', 'Age']


# Resultado

O servidor de previsão retornou:
 - Resposta do servidor:
 {"status":"success","dev_key":"Omicron","accuracy":0.6224489795918368,"old_accuracy":0.64285714285714} 